<a href="https://colab.research.google.com/github/HAMAATHI/Statistical-Learning-e22129/blob/main/Assignment_04_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import io
import warnings
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OneHotEncoder, OrdinalEncoder
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

warnings.filterwarnings('ignore')

class PlottingMethods:
    """
    Modular plotting methods returning HTML-wrapped Plotly figures for flexible embedding.
    """

    @staticmethod
    def bar_chart(df, x, y, title="Bar Chart"):
        fig = px.bar(df, x=x, y=y, title=title)
        return fig.to_html(full_html=False)

    @staticmethod
    def pie_chart(df, names, values, title="Pie Chart"):
        fig = px.pie(df, names=names, values=values, title=title)
        return fig.to_html(full_html=False)

    @staticmethod
    def histogram(df, x, title="Histogram"):
        fig = px.histogram(df, x=x, title=title)
        return fig.to_html(full_html=False)


class DataInspector:
    """
    An end-to-end tool for CSV data ingestion, advanced cleaning,
    feature engineering preparation, and high-level statistical visualization.
    """

    def __init__(self):
        self.df = None
        self.plotter = PlottingMethods()

    # ==========================================
    # 1. Data Ingestion & Sanitization
    # ==========================================

    def upload_data(self):
        """Handles manual file uploads in Google Colab, sanitizes inputs, and auto-corrects types."""
        try:
            from google.colab import files
        except ImportError:
            print("Error: upload_data() is designed specifically for Google Colab.")
            return

        print("Please upload your CSV file...")
        uploaded = files.upload()

        if not uploaded:
            print("No file uploaded.")
            return

        filename = list(uploaded.keys())[0]
        garbage_strings = ['?', 'n/a', 'NULL', ' ', 'N/A', 'na', 'nan', 'NaN']

        # Read CSV and immediately convert garbage strings to real NaNs
        self.df = pd.read_csv(io.BytesIO(uploaded[filename]), na_values=garbage_strings)
        print(f"Successfully loaded '{filename}'. Shape: {self.df.shape}")

        self._auto_correct_types()

    def _auto_correct_types(self):
        """Force-converts object columns to numeric if it doesn't nullify the whole column."""
        for col in self.df.columns:
            if self.df[col].dtype == 'object':
                converted = pd.to_numeric(self.df[col], errors='coerce')
                # If conversion doesn't make everything NaN (unless it was already all NaN)
                if not converted.isna().all() or self.df[col].isna().all():
                    self.df[col] = converted
        print("Auto-type correction applied.")

    # ==========================================
    # 2. Structural Analysis & Cleaning
    # ==========================================

    def data_summary(self):
        """Displays row/column counts, numerical/categorical breakdown, and first 20 rows."""
        if self.df is None: return "No data loaded."

        num_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        cat_cols = self.df.select_dtypes(exclude=[np.number]).columns.tolist()

        print("="*40)
        print(f"DATA SUMMARY")
        print("="*40)
        print(f"Rows: {self.df.shape[0]} | Columns: {self.df.shape[1]}")
        print(f"Numerical Columns ({len(num_cols)}): {num_cols}")
        print(f"Categorical Columns ({len(cat_cols)}): {cat_cols}")
        print("="*40)
        display(self.df.head(20))

    def handle_missing_values(self, strategy='mean', fill_value=None):
        """Imputes missing values. Strategies: mean, median, mode, constant."""
        if self.df is None: return

        for col in self.df.columns:
            if self.df[col].isnull().sum() == 0: continue

            if strategy == 'constant' and fill_value is not None:
                self.df[col].fillna(fill_value, inplace=True)
            elif strategy == 'mode':
                self.df[col].fillna(self.df[col].mode()[0], inplace=True)
            elif self.df[col].dtype in [np.float64, np.int64]:
                if strategy == 'mean':
                    self.df[col].fillna(self.df[col].mean(), inplace=True)
                elif strategy == 'median':
                    self.df[col].fillna(self.df[col].median(), inplace=True)
            else:
                # Fallback for categoricals if mean/median requested
                self.df[col].fillna(self.df[col].mode()[0], inplace=True)
        print(f"Missing values handled using '{strategy}' strategy.")

    def remove_duplicates(self):
        """Prunes exact row matches."""
        initial_len = len(self.df)
        self.df.drop_duplicates(inplace=True)
        print(f"Removed {initial_len - len(self.df)} duplicate rows.")

    def handle_outliers(self, column, action='flag'):
        """IQR-based outlier management. Action can be 'flag' or 'drop'."""
        if self.df[column].dtype not in [np.float64, np.int64]:
            print(f"Skipping {column}: Not a numeric column.")
            return

        Q1 = self.df[column].quantile(0.25)
        Q3 = self.df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outlier_condition = (self.df[column] < lower_bound) | (self.df[column] > upper_bound)

        if action == 'drop':
            initial_len = len(self.df)
            self.df = self.df[~outlier_condition]
            print(f"Dropped {initial_len - len(self.df)} outliers in '{column}'.")
        elif action == 'flag':
            self.df[f'{column}_is_outlier'] = outlier_condition
            print(f"Flagged {outlier_condition.sum()} outliers in '{column}'.")

    def delete_columns(self):
        """Interactive method to delete columns."""
        cols = input("Enter columns to delete (comma-separated): ")
        cols_to_drop = [c.strip() for c in cols.split(',') if c.strip() in self.df.columns]
        if cols_to_drop:
            self.df.drop(columns=cols_to_drop, inplace=True)
            print(f"Dropped columns: {cols_to_drop}")

    def delete_rows(self):
        """Interactive method to delete rows by index."""
        rows = input("Enter row indices to delete (comma-separated): ")
        try:
            rows_to_drop = [int(r.strip()) for r in rows.split(',') if int(r.strip()) in self.df.index]
            if rows_to_drop:
                self.df.drop(index=rows_to_drop, inplace=True)
                print(f"Dropped {len(rows_to_drop)} rows.")
        except ValueError:
            print("Invalid input. Please enter numeric indices.")

    # ==========================================
    # 3. Feature Engineering Preparation
    # ==========================================

    def extract_normalized_numeric_data(self, strategy='standard'):
        """Scales numeric data based on minmax, standard, or robust."""
        num_cols = self.df.select_dtypes(include=[np.number]).columns
        scaler_map = {
            'minmax': MinMaxScaler(),
            'standard': StandardScaler(),
            'robust': RobustScaler()
        }

        if strategy not in scaler_map: return None

        scaler = scaler_map[strategy]
        scaled_data = scaler.fit_transform(self.df[num_cols].dropna())
        return pd.DataFrame(scaled_data, columns=[f"{c}_scaled" for c in num_cols], index=self.df[num_cols].dropna().index)

    def extract_normalized_categorical_data(self, strategy='onehot'):
        """Encodes categorical data. Strategies: onehot, ordinal, uniform."""
        cat_cols = self.df.select_dtypes(exclude=[np.number]).columns
        if len(cat_cols) == 0: return pd.DataFrame()

        if strategy == 'onehot':
            encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
            encoded = encoder.fit_transform(self.df[cat_cols].fillna('Missing'))
            return pd.DataFrame(encoded, columns=encoder.get_feature_names_out(cat_cols), index=self.df.index)

        elif strategy == 'ordinal':
            encoder = OrdinalEncoder()
            encoded = encoder.fit_transform(self.df[cat_cols].fillna('Missing'))
            return pd.DataFrame(encoded, columns=[f"{c}_ordinal" for c in cat_cols], index=self.df.index)

        elif strategy == 'uniform':
            # Encoded uniformly between 0 and 1
            encoder = OrdinalEncoder()
            encoded = encoder.fit_transform(self.df[cat_cols].fillna('Missing'))
            scaler = MinMaxScaler()
            uniform_encoded = scaler.fit_transform(encoded)
            return pd.DataFrame(uniform_encoded, columns=[f"{c}_uniform" for c in cat_cols], index=self.df.index)

    def merge_normalized_data(self, num_strategy='standard', cat_strategy='onehot'):
        """Creates a unified DataFrame with scaled numerics and encoded categoricals."""
        num_df = self.extract_normalized_numeric_data(num_strategy)
        cat_df = self.extract_normalized_categorical_data(cat_strategy)
        return pd.concat([num_df, cat_df], axis=1)

    # ==========================================
    # 4. Advanced Interactive Visualization
    # ==========================================

    def univariate_subplots(self, column):
        """Generates a 3-panel subplot: Horizontal Violin/Box, Scatter, and Histogram."""
        if self.df[column].dtype not in [np.float64, np.int64]:
            print("Subplots require a numeric column.")
            return

        fig = make_subplots(rows=1, cols=3, subplot_titles=("Violin Plot", "Scatter (Index vs Value)", "Histogram"))

        fig.add_trace(go.Violin(x=self.df[column], box_visible=True, name=column, orientation='h'), row=1, col=1)
        fig.add_trace(go.Scatter(x=self.df.index, y=self.df[column], mode='markers', name=column), row=1, col=2)
        fig.add_trace(go.Histogram(x=self.df[column], name=column), row=1, col=3)

        fig.update_layout(title_text=f"Univariate Analysis: {column}", height=400, showlegend=False)
        fig.show()

    def plot_relationship(self, col1, col2):
        """Detects column types and intelligently routes to the correct chart."""
        is_num1 = self.df[col1].dtype in [np.float64, np.int64]
        is_num2 = self.df[col2].dtype in [np.float64, np.int64]

        # Num-Num: Scatter with OLS
        if is_num1 and is_num2:
            fig = px.scatter(self.df, x=col1, y=col2, trendline="ols", title=f"Scatter: {col1} vs {col2}")

        # Cat-Cat: Grouped Bar
        elif not is_num1 and not is_num2:
            df_counts = self.df.groupby([col1, col2]).size().reset_index(name='Count')
            fig = px.bar(df_counts, x=col1, y='Count', color=col2, barmode='group', title=f"Grouped Bar: {col1} vs {col2}")

        # Cat-Num: Box plot with data points
        else:
            cat_col, num_col = (col1, col2) if not is_num1 else (col2, col1)
            fig = px.box(self.df, x=cat_col, y=num_col, points="all", title=f"Box Plot: {cat_col} vs {num_col}")

        fig.show()

    def plot_categorical_frequency(self, column):
        """Bar chart displaying raw counts and percentage labels."""
        if self.df[column].dtype in [np.float64, np.int64]:
            print("Requires a categorical column.")
            return

        counts = self.df[column].value_counts().reset_index()
        counts.columns = [column, 'Count']
        counts['Percentage'] = (counts['Count'] / counts['Count'].sum() * 100).round(1).astype(str) + '%'

        fig = px.bar(counts, x=column, y='Count', text='Percentage', title=f"Frequency of {column}")
        fig.update_traces(textposition='outside')
        fig.show()

    # ==========================================
    # 5. Deep Statistical Insights
    # ==========================================

    def plot_all_associations_heatmap(self):
        """
        Visualizes relationships across all types.
        Num-Num: Pearson
        Cat-Cat: Cramer's V
        Num-Cat: Correlation Ratio (Eta)
        """
        cols = self.df.columns
        n = len(cols)
        assoc_matrix = np.zeros((n, n))

        # Helper functions inside the method
        def cramers_v(x, y):
            confusion_matrix = pd.crosstab(x, y)
            if confusion_matrix.empty: return 0.0
            chi2 = stats.chi2_contingency(confusion_matrix)[0]
            n_obs = confusion_matrix.sum().sum()
            if n_obs == 0: return 0.0
            phi2 = chi2 / n_obs
            r, k = confusion_matrix.shape
            phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n_obs-1))
            rcorr = r - ((r-1)**2)/(n_obs-1)
            kcorr = k - ((k-1)**2)/(n_obs-1)
            return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1))) if min((kcorr-1), (rcorr-1)) > 0 else 0.0

        def correlation_ratio(categories, measurements):
            fcat, _ = pd.factorize(categories)
            cat_num = np.max(fcat)+1
            y_avg_array = np.zeros(cat_num)
            n_array = np.zeros(cat_num)
            for i in range(0,cat_num):
                cat_measures = measurements[np.argwhere(fcat == i).flatten()]
                n_array[i] = len(cat_measures)
                y_avg_array[i] = np.average(cat_measures) if len(cat_measures) > 0 else 0
            y_total_avg = np.sum(np.multiply(y_avg_array,n_array))/np.sum(n_array) if np.sum(n_array) > 0 else 0
            numerator = np.sum(np.multiply(n_array,np.power(np.subtract(y_avg_array,y_total_avg),2)))
            denominator = np.sum(np.power(np.subtract(measurements,y_total_avg),2))
            if denominator == 0: return 0.0
            return np.sqrt(numerator/denominator)

        clean_df = self.df.dropna() # Drop NaNs for valid correlations

        for i in range(n):
            for j in range(n):
                if i == j:
                    assoc_matrix[i, j] = 1.0
                    continue

                col1, col2 = cols[i], cols[j]
                is_num1 = clean_df[col1].dtype in [np.float64, np.int64]
                is_num2 = clean_df[col2].dtype in [np.float64, np.int64]

                if is_num1 and is_num2:
                    assoc_matrix[i, j] = clean_df[col1].corr(clean_df[col2])
                elif not is_num1 and not is_num2:
                    assoc_matrix[i, j] = cramers_v(clean_df[col1], clean_df[col2])
                else:
                    cat_col, num_col = (col1, col2) if not is_num1 else (col2, col1)
                    assoc_matrix[i, j] = correlation_ratio(clean_df[cat_col], clean_df[num_col].values)

        fig = px.imshow(assoc_matrix, x=cols, y=cols, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                        title="Unified Association Matrix (Pearson / Cramer's V / Correlation Ratio)")
        fig.show()

In [4]:
# 1. Initialize Inspector
inspector = DataInspector()

# 2. Upload Data (Will prompt Colab upload widget)
# Tip: Upload a standard dataset like titanic.csv
inspector.upload_data()

# 3. Structural Analysis
inspector.data_summary()

# 4. Intelligent Imputation (Fill missing numerics with median, categories with mode)
inspector.handle_missing_values(strategy='median')

# 5. Outlier Handling (Flagging outliers in a numeric column, e.g., 'Fare' or 'Age')
# Replace 'Fare' with an actual numeric column from your uploaded dataset
numeric_cols = inspector.df.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 0:
    inspector.handle_outliers(column=numeric_cols[0], action='flag')

# 6. Feature Engineering (Previewing merged normalized data)
merged_features = inspector.merge_normalized_data(num_strategy='robust', cat_strategy='uniform')
print("\nPreview of Unified Normalized Data:")
display(merged_features.head())

# 7. Visualizations
if len(numeric_cols) > 0:
    # Univariate Subplot
    inspector.univariate_subplots(numeric_cols[0])

cat_cols = inspector.df.select_dtypes(exclude=[np.number]).columns
if len(cat_cols) > 0 and len(numeric_cols) > 0:
    # Smart Relationship (Cat-Num)
    inspector.plot_relationship(cat_cols[0], numeric_cols[0])

    # Categorical Frequency
    inspector.plot_categorical_frequency(cat_cols[0])

# 8. Deep Statistical Insights
inspector.plot_all_associations_heatmap()

Please upload your CSV file...


Saving dataset.csv to dataset (1).csv
Successfully loaded 'dataset (1).csv'. Shape: (48842, 15)
Auto-type correction applied.
DATA SUMMARY
Rows: 48842 | Columns: 15
Numerical Columns (6): ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Categorical Columns (9): ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'native-country', 'salary']


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,salary
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,NaN,227026,HS-grad,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


Missing values handled using 'median' strategy.
Flagged 216 outliers in 'age'.

Preview of Unified Normalized Data:


,age_scaled,fnlwgt_scaled,education-num_scaled,capital-gain_scaled,capital-loss_scaled,hours-per-week_scaled,workclass_uniform,education_uniform,marital-status_uniform,occupation_uniform,relationship_uniform,race_uniform,gender_uniform,native-country_uniform,salary_uniform,age_is_outlier_uniform
0,-0.60,0.405170,-1.000000,0.0,0.0,0.0,0.428571,0.066667,0.666667,0.461538,0.6,0.5,1.0,0.95,0.0,0.0
1,0.05,-0.735527,-0.333333,0.0,0.0,2.0,0.428571,0.733333,0.333333,0.307692,0.0,1.0,1.0,0.95,0.0,0.0
2,-0.45,1.322379,0.666667,0.0,0.0,0.0,0.142857,0.466667,0.333333,0.769231,0.0,1.0,1.0,0.95,1.0,0.0
3,0.35,-0.148399,0.000000,7688.0,0.0,0.0,0.428571,1.000000,0.333333,0.461538,0.0,0.5,1.0,0.95,1.0,0.0
4,-0.95,-0.621589,0.000000,0.0,0.0,-2.0,0.428571,1.000000,0.666667,0.692308,0.6,1.0,0.0,0.95,0.0,0.0
